In [1]:
from __future__ import annotations

import time
import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
time.sleep(1.0)

INDEX_URL = "https://www.mlit.go.jp/road/census/r3/index.html"

UA = {
    "User-Agent": "Mozilla/5.0 (student-project; traffic-census-link-extractor; contact: example.invalid)"
}

OUT_DIR = Path.cwd() / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSON = OUT_DIR / "links_all_files.json"



@dataclass
class FileLink:
    label: str     
    url: str      
    kind: str      
    ext: str      



def fetch_html(url: str) -> str:
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    return r.text


def make_soup(html: str) -> BeautifulSoup:
    try:
        return BeautifulSoup(html, "lxml")
    except Exception:
        return BeautifulSoup(html, "html.parser")


def guess_kind_ext(url: str) -> tuple[str, str]:
    u = url.lower()
    if u.endswith(".pdf"):
        return "pdf", "pdf"
    if u.endswith(".csv"):
        return "csv", "csv"
    if u.endswith(".xlsx"):
        return "excel", "xlsx"
    if u.endswith(".xls"):
        return "excel", "xls"
    return "other", ""


def extract_all_file_links(html: str, base_url: str) -> List[FileLink]:
    """
    この index.html にある PDF/Excel/CSV のリンクを全部抽出する
    （都道府県別の行はこのページには無いので、pref分類はしない）
    """
    soup = make_soup(html)

    items: List[FileLink] = []
    for a in soup.select("a[href]"):
        href = (a.get("href") or "").strip()
        if not href:
            continue

        abs_url = urljoin(base_url, href)
        kind, ext = guess_kind_ext(abs_url)
        if kind == "other":
            continue

        label = a.get_text(" ", strip=True) or abs_url.split("/")[-1]
        items.append(FileLink(label=label, url=abs_url, kind=kind, ext=ext))

    seen = set()
    uniq: List[FileLink] = []
    for it in items:
        if it.url in seen:
            continue
        seen.add(it.url)
        uniq.append(it)

    return uniq


def main():
    time.sleep(1.0)  

    html = fetch_html(INDEX_URL)

    print("contains:", "東京都" in html, "北海道" in html)

    links = extract_all_file_links(html, INDEX_URL)

    payload = {
        "source_index": INDEX_URL,
        "files_count": len(links),
        "files": [asdict(x) for x in links],
    }

    OUT_JSON.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    print("saved:", OUT_JSON)
    print("files:", payload["files_count"])
    print("\n[sample]")
    for it in links[:10]:
        print(" -", it.kind, it.url)


if __name__ == "__main__":
    main()


contains: False False
saved: /Users/fukuiyuito/Lecture/Dsprograming/dsprog2_2025/lecture-7/highway/src/data/links_all_files.json
files: 303

[sample]
 - pdf https://www.mlit.go.jp/road/census/r3/data/pdf/guide.pdf
 - pdf https://www.mlit.go.jp/road/census/r3/data/pdf/syuukeirep.pdf
 - pdf https://www.mlit.go.jp/road/census/r3/data/pdf/syuukei01.pdf
 - excel https://www.mlit.go.jp/road/census/r3/data/xlsx/syuukei01.xlsx
 - csv https://www.mlit.go.jp/road/census/r3/data/csv/syuukei01.csv
 - pdf https://www.mlit.go.jp/road/census/r3/data/pdf/syuukei02.pdf
 - excel https://www.mlit.go.jp/road/census/r3/data/xlsx/syuukei02.xlsx
 - csv https://www.mlit.go.jp/road/census/r3/data/csv/syuukei02.csv
 - pdf https://www.mlit.go.jp/road/census/r3/data/pdf/syuukei03.pdf
 - excel https://www.mlit.go.jp/road/census/r3/data/xlsx/syuukei03.xlsx


In [2]:
import csv
import time
import requests
from io import StringIO

time.sleep(1.0)
UA = {"User-Agent": "Mozilla/5.0 (student-project; contact: example.invalid)"}

def sniff_csv_header(url: str) -> list[str]:
    
    r = requests.get(url, headers=UA, timeout=30, stream=True)
    r.raise_for_status()

   
    chunk = r.raw.read(65536)
    r.close()

    for enc in ("utf-8-sig", "cp932", "utf-8"):
        try:
            text = chunk.decode(enc)
            break
        except Exception:
            text = None
    if text is None:
        return []

    # 1行目をヘッダとして読む
    buf = StringIO(text)
    reader = csv.reader(buf)
    try:
        header = next(reader)
    except StopIteration:
        return []
    return [h.strip() for h in header if h is not None]

def has_pref_column(header: list[str]) -> bool:
    keys = "".join(header)
    return any(k in keys for k in ["都道府県", "都道府県名", "都道府県コード", "府県", "県名", "地域"])


import json
from pathlib import Path

links_json = Path("data/links_all_files.json")
obj = json.loads(links_json.read_text(encoding="utf-8"))
csv_urls = [x["url"] for x in obj["files"] if x["kind"] == "csv"]

hits = []
for u in csv_urls[:60]:  
    header = sniff_csv_header(u)
    if header and has_pref_column(header):
        hits.append((u, header[:20]))

print("hit:", len(hits))
for u, h in hits[:10]:
    print("\n", u)
    print(" header:", h)


hit: 58

 https://www.mlit.go.jp/road/census/r3/data/csv/syuukei01.csv
 header: ['都道府県', '道路種別', '交通調査\n基本区間数', '道路状況調査\n単位区間数', '交通量調査\n単位区間数', 'うち交通量\n観測区間数', 'うち24時間\n観測区間数', '旅行速度調査\n単位区間数', 'うち旅行速度\n計測区間数']

 https://www.mlit.go.jp/road/census/r3/data/csv/syuukei02.csv
 header: ['都道府県', '道路種別', '交通調査基本区間数／ＤＩＤ（商業地域）', '交通調査基本区間数／ＤＩＤ（商業地域を除く）', '交通調査基本区間数／その他市街部', '交通調査基本区間数／平地部', '交通調査基本区間数／山地部', '交通調査基本区間数／合計', '延長（ｋｍ）／ＤＩＤ（商業地域）', '延長（ｋｍ）／ＤＩＤ（商業地域を除く）', '延長（ｋｍ）／その他市街部', '延長（ｋｍ）／平地部', '延長（ｋｍ）／山地部', '延長（ｋｍ）／合計']

 https://www.mlit.go.jp/road/census/r3/data/csv/syuukei03.csv
 header: ['都道府県', '道路種別', '延長（ｋｍ）／ＤＩＤ（商業地域）', '延長（ｋｍ）／ＤＩＤ（商業地域を除く）', '延長（ｋｍ）／その他市街部', '延長（ｋｍ）／平地部', '延長（ｋｍ）／山地部', '延長（ｋｍ）／合計', '歩道設置延長（ｋｍ）／ＤＩＤ（商業地域）', '歩道設置延長（ｋｍ）／ＤＩＤ（商業地域を除く）', '歩道設置延長（ｋｍ）／その他市街部', '歩道設置延長（ｋｍ）／平地部', '歩道設置延長（ｋｍ）／山地部', '歩道設置延長（ｋｍ）／合計', '自転車歩行車道設置延長（ｋｍ）／ＤＩＤ（商業地域）', '自転車歩行車道設置延長（ｋｍ）／ＤＩＤ（商業地域を除く）', '自転車歩行車道設置延長（ｋｍ）／その他市街部', '自転車歩行車道設置延長（ｋｍ）／平地部', '自転車歩行車道設置延長（ｋｍ）／山地部', '自転車歩行車道設置延長（ｋｍ）／合計']

 https:/